In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import sys

# === CONFIGURATION ===
filename = "confusion_matrices/tiny_cnn/data/img_classification_layer1_trainset"
noglitch_csv = filename + "_no_glitch.csv"

voltage_label = ""
position_label = ""

# === Initialize lists ===
percentages = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100]
percentages_labels = [x / 100 for x in range(21)]

correct_list = []
faulty_list = []
fail_list = []
intermediate_list = []

print("Starting Voltage data analysis and plotting...")

# === Load Baseline (No-Glitch) Data ===
df_noglitch = pd.read_csv(noglitch_csv)

# === Loop through each glitch percentage ===
for percentage in percentages:
    print(f"Analyzing {percentage}% glitch data...")

    path = f"{filename}_{percentage}%_run3_95e-9.csv"

    try:
        df_faulty = pd.read_csv(path)
    except FileNotFoundError:
        print(f"File not found for {percentage:.2f}% : skipping this percentage. (Tried: {path})")
        correct_list.append(np.nan)
        faulty_list.append(np.nan)
        fail_list.append(np.nan)
        intermediate_list.append(np.nan)
        continue

    # --- Basic counts ---
    total = len(df_faulty)
    correct_count = df_faulty["Correct"].sum()
    faulty_count = df_faulty["Faulty"].sum()
    fail_count = df_faulty["Fail"].sum()
    total_predicted = correct_count + faulty_count

    # --- Intermediate Value Error Detection ---
    intermediate_cols = ["CNN_out", "Maxpool_out", "FC_out", "DQ_Out"]

    valid_intermediate_df = df_faulty[
        (df_faulty["CNN_out"] != "[]") & (df_faulty["Fail"] == 0)
    ]
    valid_baseline_df = df_noglitch.loc[valid_intermediate_df.index]

    intermediate_diff_mask = (
        valid_intermediate_df[intermediate_cols] != valid_baseline_df[intermediate_cols]
    ).any(axis=1)

    intermediate_error_count = intermediate_diff_mask.sum()
    total_valid = len(valid_intermediate_df)

    # --- Compute percentages ---
    correct_pct = 100 * correct_count / total_predicted if total_predicted > 0 else 0
    faulty_pct = 100 * faulty_count / total_predicted if total_predicted > 0 else 0
    fail_pct = 100 * fail_count / total if total > 0 else 0
    intermediate_pct = 100 * intermediate_error_count / total_valid if total_valid > 0 else 0

    # --- Store in lists ---
    correct_list.append(correct_pct)
    faulty_list.append(faulty_pct)
    fail_list.append(fail_pct)
    intermediate_list.append(intermediate_pct)

# === Plotting section ===
labels = [f"{p:g}t" for p in percentages_labels]
x = np.arange(len(labels))
width = 0.18

fig, ax = plt.subplots(figsize=(12.5, 6.2))

# Match uploaded PDF bar style more closely:
# Accuracy, Error in Intermediate Value, Reset, Effective Error
ax.bar(
    x - 1.5 * width,
    correct_list,
    width,
    label="Accuracy",
    color="blue",
    edgecolor="black",
    linewidth=1.0,
    hatch="//"
)

ax.bar(
    x - 0.5 * width,
    intermediate_list,
    width,
    label="Error in Intermediate Value",
    color="yellow",
    edgecolor="black",
    linewidth=1.0,
    hatch="xx"
)

ax.bar(
    x + 0.5 * width,
    fail_list,
    width,
    label="Reset",
    color="orange",
    edgecolor="black",
    linewidth=1.0,
    hatch=".."
)

ax.bar(
    x + 1.5 * width,
    faulty_list,
    width,
    label="Effective Error",
    color="lightgray",
    edgecolor="black",
    linewidth=1.0,
    hatch="\\\\"
)

# Formatting
ax.set_ylabel("Percentage (%)", fontsize=16)
ax.set_xlabel("Injection time", fontsize=16)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=90)
ax.tick_params(axis="x", labelsize=16)
ax.tick_params(axis="y", labelsize=16)

ax.set_ylim(0, 105)
ax.set_yticks(np.arange(0, 101, 20))

ax.grid(axis="y", linestyle="--", alpha=0.5, linewidth=0.8)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_linewidth(1.0)
    spine.set_color("black")

# Keep legend placement like your original script
ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.22),
    ncol=4,
    frameon=True
)

plt.tight_layout()

# === Save plot ===
save_directory = "confusion_matrices/tiny_cnn/data/plots"
os.makedirs(save_directory, exist_ok=True)
out_file = os.path.join(save_directory, "intermediate_analysis.pdf")

fig.savefig(out_file, bbox_inches="tight")
plt.show()

print(f"Plot saved to: {out_file}")